<a href="https://colab.research.google.com/github/harprabhnanda01/week_3_submission/blob/main/Another_copy_of_Module_3_Assignment_OpenRouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Please run this cell, to install the dependencies
! pip install pypdf openai pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 21.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pinecone]



# Speak with PDF Assignment

**Assignment type:** RAG mini-app with three required functions  
**File to submit:** `pdf_ai.ipynb` (the notebook produced from this spec)  
**Language:** Python 3.x (Codio / Jupyter)  
**Time estimate:** 2–3 hours

---

## Objective

Build a **“Speak with PDF” RAG pipeline** that runs inside a Jupyter notebook and is compatible with **NBGrader**. You will implement **three functions**:

1) `parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120) -> List[str]`  
2) `embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small") -> dict`  
3) `answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini") -> str`

**Grading will use OpenAI keys.** Learners must provide valid `OPENAI_API_KEY` (and Pinecone credentials if using Pinecone).

---

## What you will build

A tiny RAG system that:
- loads a PDF,
- extracts and **chunks** text,
- builds a **vector index** from OpenAI embeddings (Pinecone recommended; in‑memory/FAISS acceptable),
- answers a natural-language query by retrieving top‑k relevant chunks and generating an answer with an OpenAI chat model.

This mirrors production flow (PDF → chunks → embeddings → index → retrieval → answer).


---

## Tools & Libraries

- `openai` (for embeddings + chat completions)
- `pypdf` **or** `pdfplumber` (choose one) for text extraction
- Vector store: **Pinecone** (recommended) via `pinecone-client`; you may alternatively use **FAISS** or a simple in‑memory index for development
- `numpy`, `re`, `math`, `hashlib` (as needed)

> You may import additional libraries if helpful. Use your judgment.

**Environment variables expected (for grading):**
- `OPENAI_API_KEY`
- If using Pinecone: `PINECONE_API_KEY`, `PINECONE_ENV` (or project config), and an index name you create in notebook (e.g., `pdf-mini-app`).



---

## Function Requirements
You'll have to implment the following functionality down below

### 1) `parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120) -> list[str]`

**Inputs**
- `pdf_path: str` — path to a readable PDF file (e.g., `data/sample.pdf`)
- `chunk_size: int` (default 800) — approximate number of characters per chunk
- `overlap: int` (default 120) — number of overlapping characters between adjacent chunks

**Output**
- `List[str]` — ordered, non-empty chunks of plain text. No chunk should exceed `chunk_size + 20` characters (allowing minor boundary variance).

**Behavioral constraints**
- Strip excessive whitespace and normalize newlines.
- Deterministic chunk boundaries given the same inputs.
- Apply overlap between chunks (except the first).

**Recommended approach**
1. Extract all PDF text into a single string using `pypdf` **or** `pdfplumber`.
2. Normalize whitespace.
3. Greedy chunk by character count with the requested overlap.



---

### 2) `embed_and_index(chunks, use_pinecone=True, model="text-embedding-3-small") -> dict`

**Purpose**  
Create embeddings with **OpenAI** and store them in a vector index (Pinecone preferred).

**Inputs**
- `chunks: List[str]` — the output of `parse_pdf_to_chunks`
- `use_pinecone: bool` — if `True`, upsert to Pinecone; if `False`, store locally (FAISS or simple in‑memory array search)
- `model: str` — OpenAI embedding model name (e.g., `"text-embedding-3-small"`)

**Output**
- `dict` with fields (minimum):
  - `"chunks"`: `List[str]`
  - `"index_backend"`: `"pinecone"` or `"local"`
  - `"index_ref"`: backend-specific handle (e.g., Pinecone index name / namespace, or local matrix + ids)
  - `"embed_model"`: embedding model used
  - `"dim"`: embedding dimensionality

**Notes**
- When `use_pinecone=True`: create (if needed) and upsert vectors with stable IDs (e.g., `doc-0`, `doc-1`, …). Keep a mapping from ID → original chunk text in the dict.
- When `use_pinecone=False`: store embeddings in memory (e.g., NumPy matrix) along with their chunk IDs for cosine‑similarity search.



---

### 3) `answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini") -> str`

**Purpose**  
Retrieve relevant chunks and generate an answer using an OpenAI chat model.

**Inputs**
- `question: str`
- `index_obj: dict` — the output from `embed_and_index`
- `top_k: int` — number of chunks to retrieve
- `gpt_model: str` — OpenAI chat model (e.g., `"gpt-4o-mini"`)

**Output**
- `str` — the final answer.

**Required behavior**
- Embed the **question** using the **same embedding model** used to build the index.
- Retrieve **top_k** chunks by cosine similarity (Pinecone query or local similarity search).
- Construct a concise prompt:
  - System: “You are a helpful assistant. Use ONLY the provided context to answer. If not answerable from context, respond: ‘I don’t know based on the document.’”
  - User: include the `question` and a `Context:` section concatenating the retrieved chunks (with simple separators).
- Call the OpenAI chat completion endpoint and **return the assistant’s message string**.



In [2]:
# This cell may be run by the grader
import os, re, math, hashlib
import numpy as np

In [3]:
import os
OPENROUTER_API_KEY = ''      # required
os.environ['PINECONE_API_KEY'] = ''      # optional (Gemini)

In [4]:
OPENAI_API_KEY = OPENROUTER_API_KEY
assert OPENAI_API_KEY, "Please set OPENAI_API_KEY"
USE_PINECONE = bool(os.getenv("PINECONE_API_KEY"))

In [5]:
from pypdf import PdfReader
from openai import OpenAI

# Initialize OpenAI client configured for OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENAI_API_KEY,
)

<center><h1>--------------- Student Code ---------------</h1></center>

### B. Implement `parse_pdf_to_chunks`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [6]:
def parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120):
    """
    Extract text from a PDF and split it into overlapping chunks.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero")

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size")

    reader = PdfReader(pdf_path)

    # Extract text from every page
    all_text = []

    for page in reader.pages:
        page_text = page.extract_text() or ""
        all_text.append(page_text)

    # Combine and normalize whitespace
    text = " ".join(all_text)
    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return []

    # Create overlapping chunks
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end == len(text):
            break

        start = end - overlap

    return chunks



---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `parse_pdf_to_chunks`.
 -  If the function is not found, you answer will be rejected by the system

### C. Implement `embed_and_index`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [7]:
def embed_and_index(
    chunks,
    use_pinecone=True,
    model="text-embedding-3-small"
):
    """
    Generate embeddings using OpenRouter and store them
    in a local NumPy vector index.

    The local index is used even if use_pinecone=True,
    so no Pinecone credentials are required.
    """

    if not chunks:
        raise ValueError("No text chunks were provided.")

    # Use the configured OpenRouter embedding model
    embedding_model = (
        model if "/" in model else "openai/" + model
    )

    all_embeddings = []

    # Process chunks in batches
    batch_size = 50

    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]

        response = client.embeddings.create(
            model=embedding_model,
            input=batch
        )

        # Keep embeddings in their original order
        batch_embeddings = sorted(
            response.data,
            key=lambda item: item.index
        )

        all_embeddings.extend(
            [item.embedding for item in batch_embeddings]
        )

    # Convert embeddings into a NumPy matrix
    embedding_matrix = np.array(
        all_embeddings,
        dtype=np.float32
    )

    # Stable chunk IDs
    ids = [f"doc-{i}" for i in range(len(chunks))]

    # Return the index information
    index_obj = {
        "chunks": list(chunks),
        "ids": ids,
        "index_backend": "local",
        "index_ref": embedding_matrix,
        "embeddings": embedding_matrix,
        "embed_model": embedding_model,
        "dim": embedding_matrix.shape[1],
        "id_to_chunk": {
            ids[i]: chunks[i] for i in range(len(chunks))
        }
    }

    return index_obj


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `embed_and_index`.
 -  If the function is not found, you answer will be rejected by the system

### D. Implement `answer_query`  **(student solution cell)**

### Please Write your answer here

Delete the `raise NotImplementedError()` when you start writing your code.

In [8]:
def answer_query(
    question,
    index_obj,
    top_k=3,
    gpt_model="gpt-4o-mini"
):
    """
    Retrieve relevant PDF chunks using cosine similarity
    and generate an answer grounded only in those chunks.
    """

    if not question.strip():
        raise ValueError("Question cannot be empty.")

    chunks = index_obj["chunks"]
    embeddings = index_obj["embeddings"]
    embedding_model = index_obj["embed_model"]

    if not chunks:
        return "I don't know based on the document."

    # Generate an embedding for the question
    query_response = client.embeddings.create(
        model=embedding_model,
        input=[question]
    )

    query_embedding = np.array(
        query_response.data[0].embedding,
        dtype=np.float32
    )

    # Calculate cosine similarity
    chunk_norms = np.linalg.norm(
        embeddings, axis=1
    )

    query_norm = np.linalg.norm(query_embedding)

    if query_norm == 0:
        raise ValueError("Question embedding has zero norm.")

    similarities = np.dot(
        embeddings, query_embedding
    ) / np.maximum(chunk_norms * query_norm, 1e-12)

    # Retrieve top-k chunks.
    # Stable tie-breaking uses original chunk order.
    top_k = min(max(1, top_k), len(chunks))

    ranked_indices = np.lexsort(
        (np.arange(len(chunks)), -similarities)
    )

    selected_indices = ranked_indices[:top_k]

    retrieved_chunks = [
        chunks[i] for i in selected_indices
    ]

    # Combine retrieved chunks into context
    context = "\n\n---\n\n".join(retrieved_chunks)

    # Use the OpenRouter chat model
    chat_model = (
        gpt_model if "/" in gpt_model
        else "openai/" + gpt_model
    )

    system_prompt = (
        "You are a helpful assistant. "
        "Use ONLY the provided context to answer. "
        "If the answer is not available in the context, "
        "respond exactly: "
        "'I don't know based on the document.'"
    )

    user_prompt = f"""
Question:
{question}

Context:
{context}

Answer using only the provided context.
"""

    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_tokens=500
    )

    return response.choices[0].message.content or ""


print("All three RAG functions have been implemented!")


All three RAG functions have been implemented!


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `answer_query`.
 -  If the function is not found, you answer will be rejected by the system


### E. (Optional) Demo cell (ungraded)
Provide a small demo that:
- Loads `data/sample.pdf`
- Runs all three functions
- Prints an example answer



In [9]:
# Let's create a mock or actual PDF file for testing, and run the whole RAG pipeline
import os

# Create a simple sample PDF file if it doesn't exist
os.makedirs("data", exist_ok=True)
pdf_path = "data/sample.pdf"

# We'll use a simple text write to simulate creating a PDF if none is present,
# but since we need a valid PDF file for PdfReader, let's write a quick placeholder.
# (If you already have a PDF in 'data/sample.pdf', this pipeline will read it directly!)
try:
    # Attempt to run a basic end-to-end test
    print("--- Starting Speak with PDF Demo ---")

    # 1. Parse PDF
    print("1. Parsing PDF to chunks...")
    # Note: Ensure you have uploaded a valid PDF to 'data/sample.pdf' for a full PDF run.
    # For testing, we can point to any local PDF file.
    chunks = parse_pdf_to_chunks(pdf_path, chunk_size=400, overlap=50)
    print(f"Extracted {len(chunks)} text chunks.")

    # 2. Embed and Index
    print("\n2. Embedding chunks & indexing locally...")
    index_obj = embed_and_index(chunks, use_pinecone=False)
    print(f"Index created successfully with dimensionality: {index_obj['dim']}")

    # 3. Answer a sample query
    question = "What is the main topic of the document?"
    print(f"\n3. Querying: '{question}'")
    answer = answer_query(question, index_obj, top_k=2)
    print(f"Answer:\n{answer}")

except Exception as e:
    print(f"Demo running note: Make sure a valid PDF is uploaded to '{pdf_path}'. Error details: {e}")

--- Starting Speak with PDF Demo ---
1. Parsing PDF to chunks...
Demo running note: Make sure a valid PDF is uploaded to 'data/sample.pdf'. Error details: [Errno 2] No such file or directory: 'data/sample.pdf'


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  If the function is not found, you answer will be rejected by the system

---

## Hidden Tests (run by grader)

Hidden tests will verify that:

- `parse_pdf_to_chunks` applies overlap correctly, is deterministic, and honors size bounds.
- `embed_and_index` calls OpenAI embeddings and builds a valid index (Pinecone or local), with stable IDs and correct dimensionality/shape.
- Retrieval uses cosine similarity and returns the expected top‑k order on crafted inputs.
- `answer_query` embeds the question with the same model used for chunks, constructs a context‑bounded prompt, calls OpenAI chat completions, and returns a clean string.
- No hard‑coded secrets; keys must be read from environment.
- Notebook runs top‑to‑bottom reproducibly when keys are present.



---

## Rubric (100 pts)

- **PDF parsing & chunking (30 pts)**
  - Correct extraction and normalization (10)
  - Deterministic overlapping chunks within size bounds (15)
  - Edge cases (empty pages, weird whitespace) (5)

- **Embeddings & indexing (35 pts)**
  - Correct OpenAI embedding usage and shapes (15)
  - Proper index structure + metadata (10)
  - Stable IDs and retrievability (10)

- **Query flow & answer (25 pts)**
  - Correct cosine similarity retrieval and stable tie‑break (10)
  - Prompt enforces context‑only answering; sensible output (15)

- **Code quality & constraints (10 pts)**
  - Clean, readable, documented code (5)
  - Secure key handling; graceful error handling (5)



---

## (Optional) Pinecone/OpenAI Notes

- For Pinecone: create an index (e.g., 384 or 1536 dim depending on embedding model) and upsert vectors with metadata (`{"chunk": "...", "order": i}`).
- For local dev without Pinecone: store embeddings in a NumPy matrix and do cosine similarity. Grading is OK with either backend as long as OpenAI embeddings + chat are used.
- Keep your mapping from vector IDs to original chunk text; you’ll need it for context construction.



---

## Submission

Submit your completed notebook. Ensure it runs top‑to‑bottom without modification in a clean environment **with valid API keys set**. The grader will run hidden tests using OpenAI for embeddings and chat.


---

<h2><center> Completing your test </center></h2>

- Once you have completed your test and answered all the questions:
  
- You can open the guide using the button located on the far right of your screen.

- The button will look like this image:  
  ![](https://github.com/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%203/additional_files/End_guide_button.png?raw=1)

- Clicking on this button will allow you to mark the test as complete using the button shown below:  
  ![](https://github.com/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%203/additional_files/completion_button.png?raw=1)

- Please click on **Mark as Completed** to end and submit your test.

- Thank you!
